In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import log_loss

In [6]:
gbm_data = pd.read_csv('gbm-data.csv')
X_features = gbm_data.iloc[:, 1:].values
y_target = gbm_data.iloc[:, 0].values
X_training, X_testing, y_training, y_testing = train_test_split(
    X_features, y_target, test_size=0.8, random_state=241
)

In [7]:
learning_rate_values = [1, 0.5, 0.3, 0.2, 0.1]
model_results = {'minimum_loss': np.inf, 'best_iteration': -1}

In [8]:
for rate in learning_rate_values:
    boosting_classifier = GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=rate,
        random_state=241,
        verbose=False
    )
    boosting_classifier.fit(X_training, y_training)
    
    # Вычисление log-loss на каждой итерации
    losses_on_test = []
    predictions_staged = boosting_classifier.staged_decision_function(X_testing)
    
    for stage_prediction in predictions_staged:
        sigmoid_output = 1.0 / (1.0 + np.exp(-stage_prediction))
        stage_loss = log_loss(y_testing, sigmoid_output)
        losses_on_test.append(stage_loss)
    
    # Фиксируем результаты для learning_rate=0.2
    if rate == 0.2:
        model_results['minimum_loss'] = min(losses_on_test)
        model_results['best_iteration'] = np.argmin(losses_on_test) + 1

# Сохранение ответов
with open('ans1.txt', 'w') as file:
    file.write('overfitting')
print("ans1: overfitting")

ans1: overfitting


In [9]:
answer_2_text = f"{model_results['minimum_loss']:.2f} {model_results['best_iteration']}"
with open('ans2.txt', 'w') as file:
    file.write(answer_2_text)
print(f"ans2: {answer_2_text}")

ans2: 0.53 37


In [11]:
forest_classifier = RandomForestClassifier(
    n_estimators=model_results['best_iteration'],
    random_state=241
)
forest_classifier.fit(X_training, y_training)
forest_probabilities = forest_classifier.predict_proba(X_testing)[:, 1]
forest_logloss = log_loss(y_testing, forest_probabilities)

answer_3_text = f"{forest_logloss:.2f}"
with open('ans3.txt', 'w') as file:
    file.write(answer_3_text)
print(f"ans3: {answer_3_text}")

ans3: 0.54
